In [2]:
import pandas as pd
import numpy as np
import requests
import sqlite3
import sqlalchemy as sqla
import os

ModuleNotFoundError: No module named 'sqlalchemy'

## Interacting with Web APIs

Many websites expose public APIs that serve data in JSON or other formats.

The `requests` package is the recommended way to make HTTP requests from Python:

```
conda install requests
```

### Making a GET Request

```python
resp = requests.get(url)
resp.raise_for_status()   # raises an error for any HTTP error status (4xx, 5xx)
```

Always call `raise_for_status()` after every request to catch HTTP errors early.

### Parsing the Response

- `resp.json()` parses the response body as JSON and returns a Python dict or list.
- A list of dicts returned by an API can be passed directly to `pd.DataFrame`.
- Use the `columns` argument to select only the fields you need.

### Note on Live API Data

Results from live APIs reflect real-time data — the output will differ each time the code is run.

## Key Points

- Use `requests.get(url)` to fetch data from a web API.
- Always call `raise_for_status()` to surface HTTP errors.
- `resp.json()` converts the response to a Python dict or list.
- A list of dicts from a JSON API can be passed directly to `pd.DataFrame`.
- Pass `columns=[...]` to select only the fields of interest.

In [ ]:
url = "https://api.github.com/repos/pandas-dev/pandas/issues"

resp = requests.get(url)
resp.raise_for_status()

resp

In [ ]:
data = resp.json()

data[0]["title"]  # most recent issue title

In [ ]:
issues = pd.DataFrame(data, columns=["number", "title", "labels", "state"])

issues

## Interacting with Databases

In business settings, data is often stored in SQL relational databases rather than files.

### Using `sqlite3` Directly

Python's built-in `sqlite3` module provides a low-level database interface:

1. Connect: `con = sqlite3.connect("file.sqlite")`
2. Execute DDL: `con.execute("CREATE TABLE ...")`
3. Insert rows: `con.executemany(stmt, list_of_tuples)`
4. Query: `cursor = con.execute("SELECT ...")` → `rows = cursor.fetchall()`
5. Build DataFrame: pass `rows` and extract column names from `cursor.description`.

`cursor.description` returns a tuple of 7-element tuples per column — only the first element (the name) is reliably populated in SQLite3.

### Using SQLAlchemy + `pd.read_sql`

SQLAlchemy abstracts over differences between SQL databases (SQLite, PostgreSQL, MySQL, etc.).

```
conda install sqlalchemy
```

`pd.read_sql(query, engine)` combines the query, fetch, and DataFrame construction into a single call — eliminating all the manual munging.

```python
db = sqla.create_engine("sqlite:///file.sqlite")
pd.read_sql("SELECT * FROM table", db)
```

## Key Points

- `sqlite3` is built into Python and works well for small local databases.
- Raw SQL results are lists of tuples — column names must be extracted from `cursor.description`.
- SQLAlchemy provides a database-agnostic connection layer.
- `pd.read_sql(query, engine)` is the clean one-liner that replaces all manual cursor work.
- The same pattern works for any database SQLAlchemy supports (SQLite, PostgreSQL, MySQL, etc.).

In [ ]:
# Create a SQLite database and table
query = """
CREATE TABLE test
(a VARCHAR(20), b VARCHAR(20),
 c REAL,        d INTEGER
);"""

con = sqlite3.connect("mydata.sqlite")
con.execute(query)
con.commit()

In [ ]:
# Insert rows
data = [
    ("Atlanta",     "Georgia",    1.25, 6),
    ("Tallahassee", "Florida",    2.6,  3),
    ("Sacramento",  "California", 1.7,  5),
]

stmt = "INSERT INTO test VALUES(?, ?, ?, ?)"
con.executemany(stmt, data)
con.commit()

In [ ]:
# Fetch results as a list of tuples
cursor = con.execute("SELECT * FROM test")
rows = cursor.fetchall()

rows

In [ ]:
# cursor.description: column names are in position [0] of each tuple
cursor.description

In [ ]:
# Build DataFrame manually from rows + cursor.description
pd.DataFrame(rows, columns=[x[0] for x in cursor.description])

In [ ]:
# SQLAlchemy + pd.read_sql: one-liner that replaces all the above
db = sqla.create_engine("sqlite:///mydata.sqlite")

pd.read_sql("SELECT * FROM test", db)

In [ ]:
con.close()
os.remove("mydata.sqlite")